# 57. 정밀도를 낮춰도 스내치 분석 결과가 유지될까?

같은 검증 선수 6명에서 원판 모델과 자세 모델의 입력을 고정합니다. 원판 박스가 맞더라도 바벨 중심 높이는 틀릴 수 있어 두 결과를 따로 봅니다. 자세는 같은 수행자의 가까운 쪽 관절만 비교합니다.

첫 셀부터 순서대로 실행하세요. Mac에서는 PyTorch와 ONNX Runtime의 FP32 대조군을 확인합니다. RTX 3090의 TensorRT 출력 4개와 실행 기록을 회수하면 같은 셀에서 FP32와 FP16을 비교합니다. 이 노트북은 로컬 파일을 읽고 로컬에만 결과를 씁니다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").exists())
sys.path.insert(0, str(root))
from wlcv.evaluation.precision_inputs import B02, prepare, source_image
from wlcv.evaluation.precision_controls import run_controls
from wlcv.evaluation.precision_quality import score

folder = root / "outputs/real_frame_precision/57_validation_v1"
if not folder.exists():
    prepare(folder)
manifest = pd.read_csv(folder / "local_frames.csv")
print("원판 검증 사진과 원본 영상 프레임을 합친 고유 입력", len(manifest), "개")
display(manifest.groupby(["source", "person"]).size().unstack(fill_value=0))

## 사용할 출력 확인

검증 사진 39장으로 mAP를 계산합니다. 원본 영상에서 b02 허브용 8장과 자세용 12장을 읽으며, 두 작업에 공통인 2장은 같은 입력을 씁니다. JPEG 검증 사진과 영상에서 직접 읽은 프레임은 픽셀이 달라질 수 있어 별개로 취급합니다.

In [ ]:
predictions = folder / "predictions"
local = ["pytorch-fp32", "ort-cpu-fp32"]
gpu = ["trt-fp32", "trt-fp16"]
models = ["plate", "pose"]

if not all((predictions / f"{r}-{m}.npz").is_file() for r in local for m in models):
    run_controls(folder)

gpu_files = [predictions / f"{r}-{m}.npz" for r in gpu for m in models]
gpu_files.append(predictions / "real_run.json")
if all(p.is_file() for p in gpu_files):
    runtimes = [local[0], *gpu]
    output = folder / "trt_quality"
    print("RTX 3090 실측. 두 TensorRT 정밀도를 비교합니다.")
elif any(p.exists() for p in gpu_files):
    raise RuntimeError("GPU 결과 회수가 일부만 끝났습니다. 파일 5개를 모두 확인하세요.")
else:
    runtimes = local
    output = folder / "local_controls"
    print("RTX 3090 실측 대기. 아래 표와 그림은 Mac의 두 FP32 결과입니다.")
tables = score(folder, runtimes)

## 원판 박스와 동작 중 미탐

mAP는 신뢰도 0.001부터 남긴 후보로 계산합니다. 직접 검출은 운영 기준인 신뢰도 0.10 이상, 정답과 IoU 0.50 이상에서 셉니다. 시간 연결 복구는 이 표의 직접 검출에 더하지 않습니다. 두 조건의 미탐 총수가 같아도 서로 다른 프레임을 놓칠 수 있어 새 미탐과 복구를 함께 확인합니다.

In [ ]:
display(tables["map"].set_index("runtime").round(6))
active = tables["photo_counts"].query("active == True")
display(active.groupby("runtime")[["tp", "fp", "fn"]].sum())
paired = pd.read_csv(output / "paired_misses.csv").query("active == True")
print("새 미탐", paired.new_fn.sum(), "개 / 복구", paired.recovered.sum(), "개")
display(paired.loc[paired[runtimes].gt(0).any(axis=1)])

## b02 허브 높이와 원판 상승량

예측 박스 중심에서 수동 허브의 y를 뺍니다. 양수는 모델 중심이 화면에서 더 아래에 있다는 뜻입니다. 이륙 20번에서 무릎 통과 30번까지 상승량은 위로 올라갈 때 양수입니다. 30~36번의 평균 절대 오차에는 정답 박스와 IoU 0.50 이상으로 대응된 프레임만 들어갑니다.

In [ ]:
hubs = tables["hubs"]
display(tables["rise"].set_index("runtime").round(3))
display(hubs.pivot(index="frame_index", columns="runtime", values=["source", "dy_px"]))
for runtime, group in hubs.query("frame_index >= 30").groupby("runtime"):
    plt.plot(group.frame_index, group.dy_px, marker="o", label=runtime)
plt.axhline(0, color="gray", linewidth=1)
plt.xlabel("b02 frame")
plt.ylabel("Box center y - manual hub y (px)")
plt.legend()
plt.show()

In [ ]:
from wlcv.annotation.plate_blur import REVIEW_FILE
from matplotlib.patches import Rectangle

frame = 35
case = manifest.query("source == 'video' and video == @B02 and frame == @frame").iloc[0]
image = source_image(case)[:, :, ::-1]
manual = pd.read_csv(REVIEW_FILE).query("video_id == @B02 and frame_index == @frame").iloc[0]
fig, axes = plt.subplots(1, len(runtimes), figsize=(4 * len(runtimes), 6))
for ax, runtime in zip(axes, runtimes):
    box = hubs.query("runtime == @runtime and frame_index == @frame").iloc[0]
    ax.imshow(image)
    ax.scatter(manual.manual_x, manual.manual_y, marker="+", c="lime", s=130, label="Manual hub")
    if pd.notna(box.cx):
        ax.add_patch(Rectangle((box.x1, box.y1), box.x2 - box.x1, box.y2 - box.y1,
                               fill=False, edgecolor="gold"))
        ax.scatter(box.cx, box.cy, marker="x", c="gold", s=55, label="Box center")
    ax.set_title(f"{runtime}\n{box.source} / IoU {box.iou:.2f}")
    ax.legend(loc="lower left")
    ax.axis("off")
fig.tight_layout()
plt.show()

## 같은 수행자의 관절 표시와 보류

이륙·무릎 통과 12장에서 기존에 확인한 사람 박스와 IoU 0.50 이상인 후보가 하나일 때 대응합니다. 가까운 쪽 네 관절은 신뢰도 0.50 기준으로 표시하거나 보류합니다. 사람이 직접 확인한 관절에만 수동점과의 거리 오차를 계산합니다. 추정한 골반이나 판독 불가 관절은 개수에 남깁니다.

In [ ]:
people = tables["performers"]
joints = tables["joints"]
display(people.groupby("runtime").matched.agg(["sum", "count"]))
display(pd.crosstab(joints.runtime, joints.status))
display(pd.crosstab(joints.loc[joints.runtime.eq(runtimes[0])].manual_status,
                    joints.loc[joints.runtime.eq(runtimes[0])].status))
display(joints.groupby(["runtime", "joint"]).manual_error_px.agg(["count", "mean"]).round(2))
changed = pd.read_csv(output / "paired_joints.csv").query("changed == True")
print("두 비교 조건에서 표시 상태가 달라진 관절", len(changed), "개")
display(changed)

In [ ]:
first, second = runtimes[-2:]
visible = joints.loc[joints.status.eq("표시")]
xy = visible.pivot(index=["video", "frame", "side", "joint"], columns="runtime", values=["x", "y"])
dx = xy["x"][second] - xy["x"][first]
dy = xy["y"][second] - xy["y"][first]
distance = np.hypot(dx, dy).dropna().sort_values(ascending=False)
print("두 조건에서 모두 표시한 점", len(distance), "개")
display(distance.head(5).rename("두 예측 사이 거리(px)").to_frame().round(3))

같은 검증 프레임을 반복해 비교한 결과입니다. b02의 상승량은 한 선수·한 구간이고, 원판이 없는 독립 영상의 오탐률은 이번 표에서 평가할 수 없습니다. 완성 영상 처리 시간은 합성 영상을 사용한 56번에서 확인합니다.